In [7]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [28]:
SKILL_KEYWORDS = [
    "python",
    "sql",
    "data+science",
    "machine+learning",
    "deep+learning",
    "statistics",
    "tableau",
    "power+bi",
    "probability",
    "data+analysis"
]

LEVELS = ["beginner", "intermediate", "expert"]

BASE_URL = "https://www.udemy.com/courses/search/?instructional_level={}&p={}&q={}&src=ukw"

MAX_PAGES = 3


In [8]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [43]:
all_courses = []
seen = set()

def scrape_page(skill, level, page):
    url = BASE_URL.format(level, page, skill.replace(" ", "+"))
    print("Opening:", url)
    driver.get(url)
    time.sleep(4)

    # each course card
    cards = driver.find_elements(By.XPATH, "//section[contains(@class,'vertical-card-module--card')]")

    for card in cards:
        try:
            title = card.find_element(By.XPATH, ".//h2//div").text

            instructor = card.find_element(
                By.XPATH,
                ".//span[@data-purpose='safely-set-inner-html:course-card:visible-instructors']"
            ).text

            try:
                short_desc = card.find_element(
                    By.XPATH,
                    ".//span[@data-purpose='safely-set-inner-html:course-card:course-headline']"
                ).text
            except:
                short_desc = ""

            try:
                rating = card.find_element(
                    By.XPATH,
                    ".//span[@data-purpose='rating-number']"
                ).text
            except:
                rating = ""

            try:
                reviews = card.find_element(
                    By.XPATH,
                    ".//div[contains(text(),'ratings')]"
                ).text
            except:
                reviews = ""

            try:
                duration = card.find_element(
                    By.XPATH,
                    ".//div[contains(text(),'total hours')]"
                ).text
            except:
                duration = ""

            try:
                level_text = card.find_element(
                    By.XPATH,
                    ".//div[text()='Beginner' or text()='Intermediate' or text()='Advanced']"
                ).text
            except:
                level_text = level.capitalize()

            try:
                card.find_element(By.XPATH, ".//div[contains(text(),'Bestseller')]")
                bestseller = "Yes"
            except:
                bestseller = "No"

            try:
                price = card.find_element(
                    By.XPATH,
                    ".//div[@data-purpose='course-price-text']"
                ).text
            except:
                price = ""

            key = title + instructor
            if key in seen:
                continue
            seen.add(key)

            all_courses.append({
                "title": title,
                "instructor": instructor,
                "short_description": short_desc,
                "rating": rating,
                "reviews": reviews,
                "duration": duration,
                "level": level_text,
                "bestseller": bestseller,
                "price": price,
                "skill_keyword": skill
            })

            print(f"Collected {len(all_courses)} : {title}")

        except Exception as e:
            print("Skipping one card due to error")
            continue


for skill in SKILL_KEYWORDS:
    for level in LEVELS:
        for page in range(1, MAX_PAGES + 1):
            print(f"\nScraping {skill} | {level} | page {page}")
            scrape_page(skill, level, page)
            time.sleep(3)



Scraping python | beginner | page 1
Opening: https://www.udemy.com/courses/search/?instructional_level=beginner&p=1&q=python&src=ukw

Scraping python | beginner | page 2
Opening: https://www.udemy.com/courses/search/?instructional_level=beginner&p=2&q=python&src=ukw
Collected 1 : Practical Python: Learn Python Basics Step by Step- Python 3
Collected 2 : 2026 Python Data Analysis & Visualization Masterclass
Collected 3 : Master Time Series Analysis and Forecasting with Python 2026
Collected 4 : Introduction to Python for Beginners
Collected 5 : Certified Python Developer
Collected 6 : AI & Python Development Megaclass - 300+ Hands-on Projects
Collected 7 : Python for Beginners with Examples
Collected 8 : Python Mastery: 100 Days, 100 Projects
Collected 9 : Python in 10 Days: The Ultimate Beginner's Bootcamp
Collected 10 : Python 3 from Beginner to Expert - Learn Python from Scratch
Collected 11 : Master Python Programming and Libraries: Python Full Course
Collected 12 : Python For Begi

In [44]:
df = pd.DataFrame(all_courses)
df.to_csv("raw_courses.csv", index=False)

In [45]:
driver.quit()
print("Done. Saved to raw_courses.csv")

Done. Saved to raw_courses.csv


In [46]:
import pandas as pd
import re

In [49]:
dd = pd.read_csv("raw_courses.csv", encoding = 'latin1')
dd.head()

,title,instructor,short_description,rating,reviews,duration,level,bestseller,price,skill_keyword
0,Practical Python: Learn Python Basics Step by ...,Edouard Renard,Get started quickly with Python (Python 3): On...,4.6,2476,4.0,Beginner,No,479,python
1,2026 Python Data Analysis & Visualization Mast...,Colt Steele,"Pandas, Matplotlib, Seaborn, & More! Analyze D...",4.6,3903,20.5,Beginner,No,629,python
2,Master Time Series Analysis and Forecasting wi...,Diogo Alves de Resende,"Time Series with Deep Learning (LSTM, TFT, N-B...",4.6,1442,37.5,Beginner,Yes,469,python
3,Introduction to Python for Beginners,Simon Sez IT,Learn Python for beginners - no prior Python p...,4.0,18,6.0,Beginner,No,449,python
4,Certified Python Developer,Pawel Krakowiak,[2026] Get Certified as a Python Developer wit...,4.4,53,NaN,Beginner,No,479,python


In [52]:
def is_english(text):
    if pd.isna(text):
        return False
    return bool(re.match(r'^[\x00-\x7F]+$', str(text)))

In [53]:
dd["combined_text"] = dd["title"].astype(str) + " " + dd["short_description"].astype(str)

mask = dd["combined_text"].apply(is_english)

df_english = dd[mask]

df_english.to_csv("english_courses.csv", index=False)

print("English-only courses saved.")

English-only courses saved.


In [1]:
import pandas as pd

In [3]:
data = pd.read_csv("raw_courses.csv", encoding='latin1')
data.head()

,title,instructor,short_description,rating,reviews,duration,level,bestseller,price,skill_keyword
0,Practical Python: Learn Python Basics Step by ...,Edouard Renard,Get started quickly with Python (Python 3): On...,4.6,2476,4.0,Beginner,No,479,python
1,2026 Python Data Analysis & Visualization Mast...,Colt Steele,"Pandas, Matplotlib, Seaborn, & More! Analyze D...",4.6,3903,20.5,Beginner,No,629,python
2,Master Time Series Analysis and Forecasting wi...,Diogo Alves de Resende,"Time Series with Deep Learning (LSTM, TFT, N-B...",4.6,1442,37.5,Beginner,Yes,469,python
3,Introduction to Python for Beginners,Simon Sez IT,Learn Python for beginners - no prior Python p...,4.0,18,6.0,Beginner,No,449,python
4,Certified Python Developer,Pawel Krakowiak,[2026] Get Certified as a Python Developer wit...,4.4,53,NaN,Beginner,No,479,python


In [5]:
print(data.shape)
print(data.columns)

(1424, 10)
Index(['title', 'instructor', 'short_description', 'rating', 'reviews',
       'duration', 'level', 'bestseller', 'price', 'skill_keyword'],
      dtype='object')


In [4]:
data.isnull().sum()

title                  0
instructor             0
short_description    291
rating                 0
reviews              124
duration             149
level                  0
bestseller             0
price                  0
skill_keyword          0
dtype: int64

In [1]:
import pandas as pd

In [7]:
data = pd.read_csv('english_courses.csv', encoding='latin1')


In [27]:
data.head()

,title,instructor,short_description,rating,reviews,duration,level,bestseller,price,skill_keyword,combined_text
0,Practical Python: Learn Python Basics Step by ...,Edouard Renard,Get started quickly with Python (Python 3): On...,4.6,2476.0,4.0,Beginner,0,479,python,Practical Python: Learn Python Basics Step by ...
1,2026 Python Data Analysis & Visualization Mast...,Colt Steele,"Pandas, Matplotlib, Seaborn, & More! Analyze D...",4.6,3903.0,20.5,Beginner,0,629,python,2026 Python Data Analysis & Visualization Mast...
2,Master Time Series Analysis and Forecasting wi...,Diogo Alves de Resende,"Time Series with Deep Learning (LSTM, TFT, N-B...",4.6,1442.0,37.5,Beginner,1,469,python,Master Time Series Analysis and Forecasting wi...
3,Introduction to Python for Beginners,Simon Sez IT,Learn Python for beginners - no prior Python p...,4.0,18.0,6.0,Beginner,0,449,python,Introduction to Python for Beginners Learn Pyt...
4,Certified Python Developer,Pawel Krakowiak,[2026] Get Certified as a Python Developer wit...,4.4,53.0,11.5,Beginner,0,479,python,Certified Python Developer [2026] Get Certifie...


In [15]:
data.isnull().sum()

title                  0
instructor             0
short_description    267
rating                 0
reviews                0
duration               0
level                  0
bestseller             0
price                  0
skill_keyword          0
combined_text          0
dtype: int64

In [10]:
data['reviews']=data['reviews'].fillna(0)

In [14]:
data.duration = data.groupby('skill_keyword')['duration'].transform(lambda x:x.fillna(x.median))

In [16]:
data.bestseller = data.bestseller.map({'Yes':1, 'No':0})

In [35]:
data.level_num.value_counts()

level_num
1    493
2    460
3    328
Name: count, dtype: int64

In [21]:
data['duration'] = pd.to_numeric(data['duration'], errors='coerce')


In [25]:
data['duration'] = data.groupby('skill_keyword')['duration'] \
                   .transform(lambda x: x.fillna(x.median()))


In [26]:
data['duration'].isna().sum()


0

In [36]:
data.head()

,title,instructor,short_description,rating,reviews,duration,level,bestseller,price,skill_keyword,combined_text,level_num
0,Practical Python: Learn Python Basics Step by ...,Edouard Renard,Get started quickly with Python (Python 3): On...,4.6,2476.0,4.0,Beginner,0,479,python,Practical Python: Learn Python Basics Step by ...,1
1,2026 Python Data Analysis & Visualization Mast...,Colt Steele,"Pandas, Matplotlib, Seaborn, & More! Analyze D...",4.6,3903.0,20.5,Beginner,0,629,python,2026 Python Data Analysis & Visualization Mast...,1
2,Master Time Series Analysis and Forecasting wi...,Diogo Alves de Resende,"Time Series with Deep Learning (LSTM, TFT, N-B...",4.6,1442.0,37.5,Beginner,1,469,python,Master Time Series Analysis and Forecasting wi...,1
3,Introduction to Python for Beginners,Simon Sez IT,Learn Python for beginners - no prior Python p...,4.0,18.0,6.0,Beginner,0,449,python,Introduction to Python for Beginners Learn Pyt...,1
4,Certified Python Developer,Pawel Krakowiak,[2026] Get Certified as a Python Developer wit...,4.4,53.0,11.5,Beginner,0,479,python,Certified Python Developer [2026] Get Certifie...,1


In [33]:
level_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Expert": 3
}

data['level_num'] = data['level'].map(level_map)

In [37]:
data.to_csv('final_data.csv')

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 7)


In [3]:
df = pd.read_csv("data-10-02.csv")
titles = df['title'].tolist()
instructors = df['instructor'].tolist()


In [4]:
titles = titles[:5]
instructors = instructors[:5]

In [9]:
from urllib.parse import quote_plus

In [16]:
results = []

for idx, title in enumerate(titles):

    print(f"Processing {idx+1}/{len(titles)}: {title}")

    try:
        query = quote_plus(title)
        search_url = f"https://www.udemy.com/courses/search/?q={query}"

        driver.get(search_url)
        time.sleep(4)  # give page time to load

        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "div[class*='course-product-card--card']")
        ))

        # FIRST COURSE CARD ONLY
        card = driver.find_element(By.CSS_SELECTOR, "div[class*='course-product-card--card']")

        # COURSE URL
        course_url = card.find_element(By.TAG_NAME, "a").get_attribute("href")

        # THUMBNAIL
        thumbnail_url = card.find_element(By.TAG_NAME, "img").get_attribute("src")

        # SCRAPED TITLE
        scraped_title = card.find_element(By.TAG_NAME, "h3").text

        # SCRAPED INSTRUCTOR
        scraped_instructor = card.find_element(
            By.CSS_SELECTOR,
            "div[data-testid='safely-set-inner-html:course-card:visible-instructors']"
        ).text

        results.append({
            "original_title": title,
            "scraped_title": scraped_title,
            "scraped_instructor": scraped_instructor,
            "course_url": course_url,
            "thumbnail_url": thumbnail_url
        })

    except Exception as e:
        print("Failed:", e)

        results.append({
            "original_title": title,
            "scraped_title": None,
            "scraped_instructor": None,
            "course_url": None,
            "thumbnail_url": None
        })

    time.sleep(6)  # slow down to avoid block

Processing 1/5: Practical Python: Learn Python Basics Step by Step- Python 3
Failed: Message: invalid session id
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e15ff3d5
	0x7ff7e15ff430
	0x7ff7e13a0ea6
	0x7ff7e13ebb13
	0x7ff7e14257a2
	0x7ff7e141fbd6
	0x7ff7e141eb41
	0x7ff7e13699e5
	0x7ff7e18db470
	0x7ff7e18d586d
	0x7ff7e18f621a
	0x7ff7e161b235
	0x7ff7e1623a5c
	0x7ff7e13687aa
	0x7ff7e1a372c8
	0x7ff93118e8d7
	0x7ff9333cc53c

Processing 2/5: 2026 Python Data Analysis & Visualization Masterclass
Failed: Message: invalid session id
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e15ff3d5
	0x7ff7e15ff430
	0x7ff7e13a0ea6
	0x7ff7e13ebb13
	0x7ff7e14257a2
	0x7ff7e141fbd6
	0x7ff7e141eb41
	0x7ff7e13699e5
	0x7ff7e18db470
	0x7ff7e18d586d
	0x7ff7e18f621a
	0x7ff7e161b235
	0x7ff7e1623a5c
	0x7ff7e13687aa
	0x7ff7e1a372c8
	0x7ff93118e8d7
	0x7ff9333cc53c

Processing 3/5: Master Time Series Analysis and Forecasting with Python 2026
Failed: Message: invalid ses

In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)


In [17]:
from bs4 import BeautifulSoup

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

first_card = soup.select_one("div[class*='course-product-card--card']")

href = first_card.find("a")["href"]
img = first_card.find("img")["src"]


AttributeError: 'NoneType' object has no attribute 'find'

In [18]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import quote_plus

In [19]:
df = pd.read_csv("data-10-02.csv")
titles = df["title"].tolist()

In [20]:
titles = titles[:5]

In [21]:
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=chrome_options)

In [22]:
results = []

for idx, title in enumerate(titles):

    print(f"Processing {idx+1}/{len(titles)}: {title}")

    try:
        query = quote_plus(title)
        search_url = f"https://www.udemy.com/courses/search/?q={query}"

        driver.get(search_url)

        # Wait for full render
        time.sleep(5)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        # Find first course card
        first_card = soup.select_one("div[class*='course-product-card--card']")

        if first_card:

            # URL
            a_tag = first_card.find("a", href=True)
            course_url = "https://www.udemy.com" + a_tag["href"] if a_tag else None

            # Thumbnail
            img_tag = first_card.find("img")
            thumbnail_url = img_tag["src"] if img_tag else None

            # Scraped Title
            h3_tag = first_card.find("h3")
            scraped_title = h3_tag.get_text(strip=True) if h3_tag else None

            # Scraped Instructor
            instructor_tag = first_card.find("div", attrs={"data-testid": "safely-set-inner-html:course-card:visible-instructors"})
            scraped_instructor = instructor_tag.get_text(strip=True) if instructor_tag else None

        else:
            course_url = None
            thumbnail_url = None
            scraped_title = None
            scraped_instructor = None

        results.append({
            "original_title": title,
            "scraped_title": scraped_title,
            "scraped_instructor": scraped_instructor,
            "course_url": course_url,
            "thumbnail_url": thumbnail_url
        })

    except Exception as e:
        print("Failed:", e)

        results.append({
            "original_title": title,
            "scraped_title": None,
            "scraped_instructor": None,
            "course_url": None,
            "thumbnail_url": None
        })

    time.sleep(4)  # slow down to avo

Processing 1/5: Practical Python: Learn Python Basics Step by Step- Python 3
Processing 2/5: 2026 Python Data Analysis & Visualization Masterclass
Processing 3/5: Master Time Series Analysis and Forecasting with Python 2026
Processing 4/5: Introduction to Python for Beginners
Processing 5/5: Certified Python Developer


In [23]:
metadata_df = pd.DataFrame(results)
metadata_df.to_csv("title_based_metadata.csv", index=False)

driver.quit()

print("Done.")

Done.


In [24]:
print(html[:2000])

<html lang="en-us"><head>
        <title>Search results | Udemy</title>
        
            
        

        
            
            


    <!-- OneTrust Cookies Consent Notice start for *.udemy.com display_type is web-->
    <script async="" src="https://www.datadoghq-browser-agent.com/us1/v4/datadog-rum.js"></script><script nonce="c4ZtqsUFAB4FmPHjL3tmzsndk+qhAE3l7l6WwB2EYiY=" type="text/javascript">
        // Note: must be called before google tag manager is initialized in OptanonWrapper()
        if (typeof window.dataLayer == 'undefined') {
            window.dataLayer = window.dataLayer || [];
        }

        function callGtag() {
            window.dataLayer.push(arguments);
        }

        var GRANTED = 'granted';
        var DENIED = 'denied';

        window.optOutConsent = {
            analytics_storage: GRANTED,
            functionality_storage: GRANTED,
            personalization_storage: GRANTED,
            security_storage: GRANTED,
            ad_storage:

In [26]:
pip install playwright


   ---------------------------------------- 0.0/36.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.8 MB ? eta -:--:--
   - -------------------------------------- 1.0/36.8 MB 3.1 MB/s eta 0:00:12
   - -------------------------------------- 1.8/36.8 MB 3.5 MB/s eta 0:00:11
   -- ------------------------------------- 2.6/36.8 MB 3.6 MB/s eta 0:00:10
   --- ------------------------------------ 3.7/36.8 MB 3.9 MB/s eta 0:00:09
   ----- ---------------------------------- 4.7/36.8 MB 4.0 MB/s eta 0:00:09
   ------ --------------------------------- 5.8/36.8 MB 4.1 MB/s eta 0:00:08
   ------- -------------------------------- 6.6/36.8 MB 4.2 MB/s eta 0:00:08
   ------- -------------------------------- 7.3/36.8 MB 4.0 MB/s eta 0:00:08
   -------- ------------------------------- 8.1/36.8 MB 4.1 MB/s eta 0:00:08
   --------- ------------------------------ 9.2/36.8 MB 4.1 MB/s eta 0:00:07
   ---------- ----------------------------- 10.0/36.8 MB 4.1 MB/s eta 0:00:07
   ---------


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: C:\Users\gauta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [34]:
import pandas as pd
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

import random
from playwright.async_api import async_playwright

In [35]:
async def scrape_udemy():

    
    results = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)  # set True after testing
        context = await browser.new_context()
        page = await context.new_page()

        for title in titles:
            try:
                print(f"Searching: {title}")

                # Search URL
                search_url = f"https://www.udemy.com/courses/search/?q={title}"
                await page.goto(search_url, timeout=60000)
                await page.wait_for_timeout(random.randint(3000, 5000))

                # Click first course using more stable selector
                first_course = page.locator('[data-purpose="search-course-card-title"]').first
                await first_course.click()
                await page.wait_for_load_state("networkidle")
                await page.wait_for_timeout(random.randint(3000, 5000))

                course_url = page.url

                # Thumbnail (course image)
                thumbnail = await page.locator("img").first.get_attribute("src")

                # Instructor name
                instructor = await page.locator('[data-purpose="instructor-name-top"]').first.inner_text()

                results.append({
                    "title": title,
                    "url": course_url,
                    "thumbnail": thumbnail,
                    "instructor": instructor
                })

                print("Done ✅")

                await page.go_back()
                await page.wait_for_timeout(random.randint(2000, 4000))

            except Exception as e:
                print("Error:", e)

        await browser.close()

    result_df = pd.DataFrame(results)
    result_df.to_csv("udemy_results.csv", index=False)
    print("Saved to udemy_results.csv 🚀")


# RUN THIS IN JUPYTER
await scrape_udemy()

Task exception was never retrieved
future: <Task finished name='Task-9' coro=<Connection.run() done, defined at C:\Users\gauta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\gauta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "C:\Users\gauta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "C:\Users\gauta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\playwright\_impl\_transport.py", line 120, in co

NotImplementedError: 

In [36]:
import pandas as pd
import time
import random

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [41]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless=new")  # if needed

driver = webdriver.Chrome(options=options)


In [42]:
results = []

# =============================
# LOOP
# =============================

for title in titles:
    try:
        print(f"Searching: {title}")

        search_url = f"https://www.udemy.com/courses/search/?q={title}"
        driver.get(search_url)

        # Wait until course cards appear
        wait.until(EC.presence_of_element_located(
            (By.XPATH, "//a[contains(@href,'/course/')]")
        ))

        time.sleep(random.randint(2,4))

        # Click first course link
        first_course = driver.find_element(By.XPATH, "(//a[contains(@href,'/course/')])[1]")
        first_course.click()

        # Wait until course page loads
        wait.until(EC.presence_of_element_located(
            (By.XPATH, "//h1")
        ))

        time.sleep(random.randint(2,4))

        course_url = driver.current_url

        # Thumbnail (course image on course page)
        try:
            thumbnail = driver.find_element(By.XPATH, "//img[contains(@src,'course')]").get_attribute("src")
        except:
            thumbnail = None

        # Instructor name
        try:
            instructor = driver.find_element(By.XPATH, "//a[contains(@href,'/user/')]").text
        except:
            instructor = None

        results.append({
            "title": title,
            "url": course_url,
            "thumbnail": thumbnail,
            "instructor": instructor
        })

        print("Done ✅")

    except Exception as e:
        print("Error:", e)

# =============================
# SAVE
# =============================

driver.quit()

pd.DataFrame(results).to_csv("udemy_results.csv", index=False)

print("Saved to udemy_results.csv 🚀")

Searching: Practical Python: Learn Python Basics Step by Step- Python 3
Error: Message: invalid session id
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e15ff3d5
	0x7ff7e15ff430
	0x7ff7e13a0ea6
	0x7ff7e13ebb13
	0x7ff7e14257a2
	0x7ff7e141fbd6
	0x7ff7e141eb41
	0x7ff7e13699e5
	0x7ff7e18db470
	0x7ff7e18d586d
	0x7ff7e18f621a
	0x7ff7e161b235
	0x7ff7e1623a5c
	0x7ff7e13687aa
	0x7ff7e1a372c8
	0x7ff93118e8d7
	0x7ff9333cc53c

Searching: 2026 Python Data Analysis & Visualization Masterclass
Error: Message: invalid session id
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e15ff3d5
	0x7ff7e15ff430
	0x7ff7e13a0ea6
	0x7ff7e13ebb13
	0x7ff7e14257a2
	0x7ff7e141fbd6
	0x7ff7e141eb41
	0x7ff7e13699e5
	0x7ff7e18db470
	0x7ff7e18d586d
	0x7ff7e18f621a
	0x7ff7e161b235
	0x7ff7e1623a5c
	0x7ff7e13687aa
	0x7ff7e1a372c8
	0x7ff93118e8d7
	0x7ff9333cc53c

Searching: Master Time Series Analysis and Forecasting with Python 2026
Error: Message: invalid session id
Stacktrace

In [44]:
import pandas as pd
import requests
import time
import random

df = pd.read_csv("data-10-02.csv")  # must contain column 'title'

results = []

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

for title in titles:
    try:
        print(f"Searching: {title}")

        search_url = "https://www.udemy.com/api-2.0/search-courses/"

        params = {
            "q": title,
            "page": 1,
            "page_size": 1
        }

        response = requests.get(search_url, headers=headers, params=params)

        data = response.json()

        if data["results"]:
            course = data["results"][0]

            results.append({
                "title": title,
                "url": "https://www.udemy.com" + course["url"],
                "thumbnail": course.get("image_480x270"),
                "instructor": course.get("visible_instructors", [{}])[0].get("display_name")
            })

            print("Done ✅")

        else:
            print("No results found")

        time.sleep(random.randint(2,4))

    except Exception as e:
        print("Error:", e)

pd.DataFrame(results).to_csv("udemy_results.csv", index=False)

print("Saved to udemy_results.csv 🚀")


Searching: Practical Python: Learn Python Basics Step by Step- Python 3
Error: Expecting value: line 1 column 1 (char 0)
Searching: 2026 Python Data Analysis & Visualization Masterclass
Error: Expecting value: line 1 column 1 (char 0)
Searching: Master Time Series Analysis and Forecasting with Python 2026
Error: Expecting value: line 1 column 1 (char 0)
Searching: Introduction to Python for Beginners
Error: Expecting value: line 1 column 1 (char 0)
Searching: Certified Python Developer
Error: Expecting value: line 1 column 1 (char 0)
Saved to udemy_results.csv 🚀


In [47]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 1.9 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 1.8 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 1.6 MB/s  0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 26.0
    Uninstalling pip-26.0:
      Successfully uninstalled pip-26.0
Note: you may need to restart the kernel to use updated packages.


In [49]:
import undetected_chromedriver as uc

options = uc.ChromeOptions()
options.add_argument("--start-maximized")

driver = uc.Chrome(
    version_main=144,   # force correct version
    options=options
)


In [50]:


results = []

# =============================
# LOOP
# =============================

for title in titles:
    try:
        print(f"Searching: {title}")

        search_url = f"https://www.udemy.com/courses/search/?q={title}"
        driver.get(search_url)

        time.sleep(random.randint(5,8))

        # Wait for course cards
        wait.until(EC.presence_of_element_located(
            (By.XPATH, "(//a[contains(@href,'/course/')])[1]")
        ))

        first_course = driver.find_element(By.XPATH, "(//a[contains(@href,'/course/')])[1]")
        driver.execute_script("arguments[0].scrollIntoView();", first_course)
        time.sleep(2)
        first_course.click()

        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        time.sleep(random.randint(4,7))

        course_url = driver.current_url

        # Thumbnail
        try:
            thumbnail = driver.find_element(By.XPATH, "//img[contains(@src,'course')]").get_attribute("src")
        except:
            thumbnail = None

        # Instructor
        try:
            instructor = driver.find_element(By.XPATH, "//a[contains(@href,'/user/')]").text
        except:
            instructor = None

        results.append({
            "title": title,
            "url": course_url,
            "thumbnail": thumbnail,
            "instructor": instructor
        })

        print("Done ✅")

        time.sleep(random.randint(3,6))

    except Exception as e:
        print("Error:", e)
        continue

driver.quit()

pd.DataFrame(results).to_csv("udemy_results.csv", index=False)

print("Saved to udemy_results.csv 🚀")


Searching: Practical Python: Learn Python Basics Step by Step- Python 3
Error: Message: invalid session id
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e15ff3d5
	0x7ff7e15ff430
	0x7ff7e13a0ea6
	0x7ff7e13ebb13
	0x7ff7e14257a2
	0x7ff7e141fbd6
	0x7ff7e141eb41
	0x7ff7e13699e5
	0x7ff7e18db470
	0x7ff7e18d586d
	0x7ff7e18f621a
	0x7ff7e161b235
	0x7ff7e1623a5c
	0x7ff7e13687aa
	0x7ff7e1a372c8
	0x7ff93118e8d7
	0x7ff9333cc53c

Searching: 2026 Python Data Analysis & Visualization Masterclass
Error: Message: invalid session id
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e15ff3d5
	0x7ff7e15ff430
	0x7ff7e13a0ea6
	0x7ff7e13ebb13
	0x7ff7e14257a2
	0x7ff7e141fbd6
	0x7ff7e141eb41
	0x7ff7e13699e5
	0x7ff7e18db470
	0x7ff7e18d586d
	0x7ff7e18f621a
	0x7ff7e161b235
	0x7ff7e1623a5c
	0x7ff7e13687aa
	0x7ff7e1a372c8
	0x7ff93118e8d7
	0x7ff9333cc53c

Searching: Master Time Series Analysis and Forecasting with Python 2026
Error: Message: invalid session id
Stacktrace

In [ ]:
import time
import random
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =============================
# CONFIG
# =============================

SKILL_KEYWORDS = [
    "python",
    "sql",
    "data science",
    "machine learning",
    "deep learning",
    "statistics",
    "tableau",
    "power bi",
    "probability",
    "data analysis"
]

LEVELS = ["beginner", "intermediate", "expert"]

BASE_URL = "https://www.udemy.com/courses/search/?instructional_level={}&p={}&q={}&src=ukw"

MAX_PAGES = 3


# =============================
# DRIVER SETUP
# =============================

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_courses = []
seen = set()


# =============================
# SCRAPER FUNCTION
# =============================

def scrape_page(skill, level, page):
    url = BASE_URL.format(level, page, skill.replace(" ", "+"))
    print("Opening:", url)

    driver.get(url)

    # Wait until course cards load
    wait.until(EC.presence_of_element_located(
        (By.XPATH, "//a[contains(@href,'/course/')]")
    ))

    time.sleep(random.randint(3, 5))

    cards = driver.find_elements(By.XPATH, "//a[contains(@href,'/course/')]")

    for card in cards:
        try:
            # Course URL
            course_url = card.get_attribute("href")

            # Thumbnail
            try:
                img = card.find_element(By.XPATH, ".//img")
                thumbnail = img.get_attribute("src") or img.get_attribute("data-src")
            except:
                thumbnail = ""

            # Title
            try:
                title = card.find_element(By.XPATH, ".//h3 | .//h2").text
            except:
                continue

            # Instructor
            try:
                instructor = card.find_element(
                    By.XPATH,
                    ".//span[contains(@class,'instructor')]"
                ).text
            except:
                instructor = ""

            # Rating
            try:
                rating = card.find_element(
                    By.XPATH,
                    ".//span[contains(@aria-label,'Rating')]"
                ).text
            except:
                rating = ""

            # Price
            try:
                price = card.find_element(
                    By.XPATH,
                    ".//div[contains(@data-purpose,'course-price-text')]"
                ).text
            except:
                price = ""

            key = title + instructor
            if key in seen:
                continue
            seen.add(key)

            all_courses.append({
                "title": title,
                "instructor": instructor,
                "rating": rating,
                "price": price,
                "course_url": course_url,
                "thumbnail": thumbnail,
                "skill_keyword": skill,
                "level": level
            })

            print(f"Collected {len(all_courses)} : {title}")

        except:
            continue


# =============================
# MAIN LOOP
# =============================

for skill in SKILL_KEYWORDS:
    for level in LEVELS:
        for page in range(1, MAX_PAGES + 1):
            print(f"\nScraping {skill} | {level} | page {page}")
            scrape_page(skill, level, page)
            time.sleep(random.randint(2, 4))


# =============================
# SAVE DATA
# =============================

driver.quit()

df = pd.DataFrame(all_courses)
df.to_csv("udemy_scraped_courses.csv", index=False)

print("\nSaved to udemy_scraped_courses.csv 🚀")
print("Total courses collected:", len(all_courses))


In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Setup Selenium WebDriver ---
# This sets up Chrome to run in "headless" mode, meaning you won't see the browser window.
# It's faster and cleaner for automated tasks. Comment out the 'options' lines if you want to see the browser.
options = webdriver.ChromeOptions()
options.add_argument("--headless=new") # Run in background
# options.add_argument("--no-sandbox")
# options.add_argument("--disable-dev-shm-usage")

# Automatically download and manage the ChromeDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [ ]:


# --- The Main Search Function ---
def get_first_course_details(search_query):
    """Navigates to the website, performs a search, and gets the first result's URL and thumbnail."""
    
    # Navigate to the course website's homepage
    # !!! REPLACE THIS WITH THE ACTUAL WEBSITE URL !!!
    driver.get("https://www.example-courses-site.com") 
    
    try:
        # 1. Find the search box and perform the search
        # How to find the right 'selector': 
        # Right-click on the search box on the webpage -> "Inspect" -> find its 'id', 'name', or 'class' attribute.
        # Common selectors are: By.NAME("q"), By.ID("search"), By.CLASS_NAME("search-field")
        search_box = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, "q")) # <--- ADJUST THIS SELECTOR
        )
        search_box.clear()
        search_box.send_keys(search_query)
        search_box.send_keys(Keys.RETURN)
        
        # 2. Wait for the search results page to load and find the first result container
        # We need a selector that identifies each course item on the search results page.
        # For example, it might be a <div> with a class like "course-card" or "result-item".
        # We wait for at least one of these elements to appear.
        first_result = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.course-card")) # <--- ADJUST THIS SELECTOR
        )
        
        # 3. From the first result, extract the course page URL
        # The link to the course is usually inside an <a> tag.
        # We find that <a> tag within the first result and get its 'href' attribute.
        link_element = first_result.find_element(By.CSS_SELECTOR, "a.course-link") # <--- ADJUST THIS SELECTOR
        course_url = link_element.get_attribute("href")
        
        # 4. From the first result, extract the thumbnail image URL
        # The thumbnail is usually an <img> tag.
        # We find it within the first result and get its 'src' attribute.
        # Some sites use 'data-src' for lazy loading, so you might need to check for that too.
        img_element = first_result.find_element(By.CSS_SELECTOR, "img.thumbnail") # <--- ADJUST THIS SELECTOR
        thumbnail_url = img_element.get_attribute("src")
        
        return course_url, thumbnail_url
        
    except Exception as e:
        print(f"Error searching for '{search_query}': {e}")
        return None, None

# --- Example usage within the main loop (from Step 2) ---
# You would integrate this loop with the function above.

# For demonstration, let's run one search
search_term = "Python for Beginners by John Smith" # Combine course and instructor name
url, thumbnail = get_first_course_details(search_term)

if url and thumbnail:
    print(f"Found URL: {url}")
    print(f"Found Thumbnail: {thumbnail}")
else:
    print("Could not find course details.")

# Don't forget to close the driver when you are completely done
# driver.quit() 

In [2]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# Setup Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")  # Run in background, remove to see browser
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# Initialize the driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
def search_udemy(course_title):
    """Search Udemy and get first course details"""
    
    # Construct search URL with your format
    # Note: Empty instructional_level and p parameters as per your URL
    search_url = f"https://www.udemy.com/courses/search/?instructional_level={{}}&p={{}}&q={course_title}&src=ukw"
    
    try:
        # Navigate to search results
        driver.get(search_url)
        
        # Wait for results to load (Udemy uses dynamic content)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-purpose='search-course-card-container']"))
        )
        
        # Find all course cards
        course_cards = driver.find_elements(By.CSS_SELECTOR, "[data-purpose='search-course-card-container']")
        
        if not course_cards:
            print(f"No results found for: {course_title}")
            return None, None, None
        
        # Get the first course card
        first_course = course_cards[0]
        
        # Extract course URL
        try:
            course_link = first_course.find_element(By.CSS_SELECTOR, "a[data-purpose='course-title-url']")
            course_url = course_link.get_attribute("href")
        except NoSuchElementException:
            course_url = None
        
        # Extract thumbnail URL
        try:
            thumbnail_img = first_course.find_element(By.CSS_SELECTOR, "img")
            thumbnail_url = thumbnail_img.get_attribute("src")
            # If src is empty, try data-src (for lazy loading)
            if not thumbnail_url or "data:image" in thumbnail_url:
                thumbnail_url = thumbnail_img.get_attribute("data-src")
        except NoSuchElementException:
            thumbnail_url = None
        
        # Extract instructor name
        try:
            instructor_element = first_course.find_element(By.CSS_SELECTOR, "[data-purpose='instructor-name']")
            instructor_name = instructor_element.text.strip()
        except NoSuchElementException:
            # Try alternative selector for instructor
            try:
                instructor_element = first_course.find_element(By.CSS_SELECTOR, ".ud-text-xs span")
                instructor_name = instructor_element.text.strip()
            except NoSuchElementException:
                instructor_name = None
        
        return course_url, thumbnail_url, instructor_name
        
    except TimeoutException:
        print(f"Timeout loading search results for: {course_title}")
        return None, None, None
    except Exception as e:
        print(f"Error processing {course_title}: {str(e)}")
        return None, None, None


In [ ]:
df = pd.read_csv('data-10-02.csv')  # Replace with your file name

# Add columns for results if they don't exist
if 'Course_URL' not in df.columns:
    df['Course_URL'] = None
if 'Thumbnail_URL' not in df.columns:
    df['Thumbnail_URL'] = None
if 'Scraped_Instructor' not in df.columns:
    df['Scraped_Instructor'] = None

# Process each course
for index, row in df.iterrows():
    course_title = row['title']  # Change 'Title' to your actual column name
    
    print(f"Processing ({index + 1}/{len(df)}): {course_title}")
    
    # Search and get details
    course_url, thumbnail_url, instructor_name = search_udemy(course_title)
    
    # Store results
    df.at[index, 'Course_URL'] = course_url
    df.at[index, 'Thumbnail_URL'] = thumbnail_url
    df.at[index, 'Scraped_Instructor'] = instructor_name
    
    # Add delay to be respectful to the server
    time.sleep(3)  # Adjust as needed

# Save results to a new Excel file
df.to_csv('courses_with_results.csv', index=False)
print("\n✅ Done! Results saved to 'courses_with_results.csv'")

# Close the browser
driver.quit()

Processing (1/1281): Practical Python: Learn Python Basics Step by Step- Python 3
Timeout loading search results for: Practical Python: Learn Python Basics Step by Step- Python 3
Processing (2/1281): 2026 Python Data Analysis & Visualization Masterclass
Timeout loading search results for: 2026 Python Data Analysis & Visualization Masterclass
Processing (3/1281): Master Time Series Analysis and Forecasting with Python 2026
Timeout loading search results for: Master Time Series Analysis and Forecasting with Python 2026
Processing (4/1281): Introduction to Python for Beginners
Timeout loading search results for: Introduction to Python for Beginners
Processing (5/1281): Certified Python Developer
